In [56]:
import os 
from dotenv import load_dotenv

#Langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings


In [57]:
load_dotenv()

True

In [58]:
groq_key = os.getenv("GROQ_API_KEY")

In [59]:
groq_key = os.getenv("GROQ_API_KEY")
JINA_key = os.getenv("JINA_API_KEY")
print("ENV VAR LOADED")

ENV VAR LOADED


LOADING DATA

In [60]:
DATA_FILE_PATH = os.path.join("data","hr_policy.txt")

DATA INGESTION

In [61]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents  = loader.load()
print("DATA LOADED")
print("=" * 40)
print(documents)


DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

#### LANGCHAIN DOCUMENT

Langchain processes everything in form of documnents
documents:
page content -- actual data
metadata - extra info about data


In [62]:
len(documents)

1

In [63]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [64]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [65]:
print(f"Toatal char in documents:{len(documents[0].page_content)}")

Toatal char in documents:2597


 SPLITTING OF DATA

In [66]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap =50
)

chunks = text_splitter.split_documents(documents)
print(chunks)
len(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

9

In [67]:
print(chunks[7].page_content)

7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


DATA EMBEDDING

In [68]:
embedding_model = JinaEmbeddings(model_name="jina-embeddings-v5-omni-small")

print("EMBEDDING MODEL IS READY, NAME IS ", embedding_model.model_name)

EMBEDDING MODEL IS READY, NAME IS  jina-embeddings-v5-omni-small


 STORE DATA IN A VECTOR BASE

In [69]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [70]:
test_query = "How many sick leaves employees get"

#similarity search

top_matches = vector_store.similarity_search(test_query,k=2)

print(f"Query: {test_query}\n")
#print(top_matches[0].page_content)
for i,match in enumerate(top_matches,start=1):
    print(f"---match{i}---")
    print(match.page_content)



Query: How many sick leaves employees get

---match1---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
---match2---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.


# TOOL

In [71]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

def search_hr_policy(question:str) -> str:
    """ 
    Search the HR policy document for information about leave, work from home,probation,
    notice period, reimbursement, code of conduct, holidays, or exit process
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunks.page_content for chunks in matching_chunks)

### DATA RETRIEVAL

LLM

In [72]:
from langchain_groq import ChatGroq

llm   = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0.7  #creativity
)

llm.model_name

'openai/gpt-oss-120b'

In [73]:
test_response = llm.invoke("hey is learning rag hard? answer in 1 line")
print(test_response.content)


Learning RAG can be challenging at first, but with clear resources and hands‑on practice it becomes manageable.


# AI AGENT

3 things:
LLM - Brain
tool - chunks
memory - no mem

In [74]:
from langchain.agents import create_agent
hr_assistant = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt = """ 
    You are a friendly HR assistant.
    Always use the search_hr_policy tool to look up facts before answering. 
    If the answer isn't in the search results, 
    say you don't know " intsead of guessing."
    """
    )
print("HR assistant is ready to answer")

HR assistant is ready to answer


In [ ]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": "tell me about holiday rules"}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [ ]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)